In [25]:
import torch
import random

In [26]:
n_way = 5
k_shot = 5
q_query = 5

In [27]:
background_root='./omniglot/python/images_background'
evaluation_root='./omniglot/python/images_evaluation'

In [28]:
from pathlib import Path

background_path = Path(background_root)
alphabet_paths = [p for p in background_path.iterdir() if p.is_dir()]
selected_alphabet_paths = random.sample(alphabet_paths, round(len(alphabet_paths) * 0.2))

train_path = background_path / 'train'
val_path = background_path / 'val'

train_root = str(train_path)
val_root = str(val_path)

if not val_path.exists():
    train_path.mkdir(exist_ok=True)
    val_path.mkdir(exist_ok=True)

    for alphabet_path in selected_alphabet_paths:
        alphabet_path.move_into(val_path)

    for alphabet_path in background_path.iterdir():
        if alphabet_path.is_dir() and alphabet_path not in [train_path, val_path]:
            alphabet_path.move_into(train_path)

In [29]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device

device(type='cuda')

In [30]:
from pathlib import Path
from typing import Iterator

from torch import Tensor
import torch.nn as nn
from torch.utils.data import IterableDataset
from torchvision.io import decode_image


class EpisodeDataset(IterableDataset):
    def __init__(self, root: str, n_way: int, k_shot: int, q_query: int, transform: nn.Module | None = None) -> None:
        super().__init__()

        self.root = root
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.transform = transform

        # self.data:list[tuple[Path,int]]=[]
        # [(WindowsPath('omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_01.png'), 0)]

        self.character_to_images: dict[str, list[str]] = {}

        self._load()

    def _load(self) -> None:
        root_path = Path(self.root)

        for alphabet_path in root_path.iterdir():
            if not alphabet_path.is_dir():
                continue

            for character_path in alphabet_path.iterdir():
                if not character_path.is_dir():
                    continue

                for image_path in character_path.iterdir():
                    if not image_path.is_file():
                        continue

                    # self.data.append((image_path,class_i))

                    self.character_to_images.setdefault(character_path.as_posix(), []).append(image_path.as_posix())

        # print(self.character_to_images[r'omniglot/python/images_background/Alphabet_of_the_Magi/character01'])

    def _load_image(self, path_str: str) -> Tensor:
        image = decode_image(path_str)

        if self.transform:
            image = self.transform(image)

        image: Tensor = image.float() / 255

        return image

    def __iter__(self) -> Iterator[tuple[Tensor, Tensor, Tensor, Tensor]]:
        while True:
            selected_character_strs = random.sample(tuple(self.character_to_images.keys()), self.n_way)

            support_list: list[Tensor] = []
            support_labels_list: list[int] = []
            query_list: list[Tensor] = []
            query_labels_list: list[int] = []

            for i, character in enumerate(selected_character_strs):
                image_strs: list[str] = random.sample(self.character_to_images[character], self.k_shot + self.q_query)

                images = [self._load_image(image_str) for image_str in image_strs]

                support_list += images[: self.k_shot]
                support_labels_list += [i] * self.k_shot
                query_list += images[self.k_shot :]
                query_labels_list += [i] * self.q_query

            support = torch.stack(support_list)  
            support_labels = torch.tensor(support_labels_list)
            query = torch.stack(query_list)
            query_labels = torch.tensor(query_labels_list)

            yield support, support_labels, query, query_labels

In [31]:
class Encoder(nn.Module):
    def __init__(self, in_channels=1, feature_channels=64, embed_dim=64) -> None:
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, feature_channels, 3, padding=1),
            nn.BatchNorm2d(feature_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(feature_channels, feature_channels, 3, padding=1),
            nn.BatchNorm2d(feature_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(feature_channels, feature_channels, 3, padding=1),
            nn.BatchNorm2d(feature_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(feature_channels, embed_dim, 3, padding=1),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)  # 输出 (batch, embed_dim, 1, 1)
        )

    def forward(self, x: Tensor):
        x = self.encoder(x)
        return x.view(x.size(0), -1)  # flatten to (batch, embed_dim)

In [32]:
class ProtoNet(nn.Module):
    def __init__(self, encoder: nn.Module) -> None:
        super().__init__()

        self.encoder: nn.Module = encoder

    def forward(self, support: Tensor, support_labels: Tensor, query: Tensor) -> Tensor:
        '''
        support: (n_way*k_shot, 1, H, W)
        support_labels: (n_way*k_shot,)
        query: (n_way*q_query, 1, H, W)

        return:
            logits: (n_way*q_query, n_way)
        '''
        # 1. embed support and query
        embed_support: Tensor = self.encoder(support)  # (n_way*k_shot, embed_dim)
        embed_query: Tensor = self.encoder(query)  # (n_way*q_query, embed_dim)

        # 2. compute prototypes
        unique_labels = torch.unique(support_labels)  # [0,1,2,...,n_way-1]
        prototypes_list = []
        for c in unique_labels:
            # mean of embeddings for class c
            embed_class = embed_support[support_labels == c]
            prototypes_list.append(embed_class.mean(0))

        prototypes: Tensor = torch.stack(prototypes_list)  # (n_way, embed_dim)

        # 3. compute negative distances as logits
        # dist: (n_query, n_way)
        dist = torch.cdist(embed_query, prototypes)

        # negative distance = higher score if closer
        logits = -dist

        return logits

In [33]:
train_set = EpisodeDataset(root=train_root, n_way=n_way, k_shot=k_shot, q_query=q_query)
val_set = EpisodeDataset(root=val_root, n_way=n_way, k_shot=k_shot, q_query=q_query)
test_set = EpisodeDataset(root=evaluation_root, n_way=n_way, k_shot=k_shot, q_query=q_query)

In [34]:
from torch.utils.data import DataLoader

train_loader: DataLoader[Tensor] = DataLoader(train_set, batch_size=None)
val_loader: DataLoader[Tensor] = DataLoader(val_set, batch_size=None)
test_loader: DataLoader[Tensor] = DataLoader(test_set, batch_size=None)

In [35]:
encoder = Encoder(in_channels=1, feature_channels=64, embed_dim=64)
model = ProtoNet(encoder).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [36]:
from tqdm import tqdm


epochs = 10
batches = 100
val_batches = 400

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for i, (support, support_labels, query, query_labels) in tqdm(enumerate(train_loader), total=batches, desc=f'Epoch {epoch}'):
        if i >= batches:
            break

        support: Tensor = support.to(device)  # (n_way*k_shot, 1, H, W)
        query: Tensor = query.to(device)  # (n_way*q_query, 1, H, W)
        support_labels: Tensor = support_labels.to(device)
        query_labels: Tensor = query_labels.to(device)

        optimizer.zero_grad()

        logits: Tensor = model(support, support_labels, query)
        loss: Tensor = loss_fn(logits, query_labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()

    val_total_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for i, (support, support_labels, query, query_labels) in enumerate(val_loader):
            if i >= val_batches:
                break

            support: Tensor = support.to(device)
            query: Tensor = query.to(device)
            support_labels: Tensor = support_labels.to(device)
            query_labels: Tensor = query_labels.to(device)

            logits: Tensor = model(support, support_labels, query)
            loss: Tensor = loss_fn(logits, query_labels)

            val_total_loss += loss.item()

            pred_labels = logits.argmax(dim=1)
            val_correct += (pred_labels == query_labels).sum().item()
            val_total += query_labels.size(0)

    val_loss = val_total_loss / val_batches
    val_accuracy = val_correct / val_total

    print(f'Train Loss: {total_loss/batches} | Val Loss: {val_loss} | Val Accuracy: {val_accuracy}')

Epoch 0: 100%|██████████| 100/100 [00:04<00:00, 21.81it/s]


Train Loss: 0.8796769958734513 | Val Loss: 0.7799450308829546 | Val Accuracy: 0.8203


Epoch 1: 100%|██████████| 100/100 [00:04<00:00, 22.34it/s]


Train Loss: 0.6680305808782577 | Val Loss: 0.64436512876302 | Val Accuracy: 0.8626


Epoch 2: 100%|██████████| 100/100 [00:04<00:00, 23.03it/s]


...

Train Loss: 0.38170066110789774 | Val Loss: 0.38066769862547517 | Val Accuracy: 0.9191


Epoch 8: 100%|██████████| 100/100 [00:04<00:00, 23.95it/s]


Train Loss: 0.33862530559301374 | Val Loss: 0.35087730107828974 | Val Accuracy: 0.9276


Epoch 9: 100%|██████████| 100/100 [00:04<00:00, 24.17it/s]


Train Loss: 0.33081389032304287 | Val Loss: 0.30800502652302386 | Val Accuracy: 0.9309


In [40]:
test_batches = 1000

model.eval()

test_total_loss = 0
test_correct = 0
test_total = 0

with torch.no_grad():
    for i, (support, support_labels, query, query_labels) in tqdm(enumerate(val_loader), total=test_batches, desc='Test'):
        if i >= test_batches:
            break

        support: Tensor = support.to(device)
        query: Tensor = query.to(device)
        support_labels: Tensor = support_labels.to(device)
        query_labels: Tensor = query_labels.to(device)

        logits: Tensor = model(support, support_labels, query)
        loss: Tensor = loss_fn(logits, query_labels)

        test_total_loss += loss.item()

        pred_labels = logits.argmax(dim=1)
        test_correct += (pred_labels == query_labels).sum().item()
        test_total += query_labels.size(0)

    test_loss = total_loss / test_batches
    test_accuracy = test_correct / test_total

    print(f'Test Loss: {test_loss} | Test Accuracy: {test_accuracy}')

Test: 100%|██████████| 1000/1000 [00:18<00:00, 54.28it/s]

Test Loss: 0.03308138903230429 | Test Accuracy: 0.93204


# TODO: Schedule and Early-stop